In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
USE CATALOG databricksdemo;
USE SCHEMA silver;

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@storageaccountsunnybest.dfs.core.windows.net/customers")
df = df.drop("_rescued_data")
display(df)

In [0]:
# Fetch customer domains from emails
df = df.withColumn("domain", split("email", "@")[1])
display(df)

In [0]:
# Get the most popular domains
df.groupBy("domain").agg(count("*").alias("count"))\
    .orderBy(col("count").desc())\
    .display()

In [0]:
df_gmail = df.filter(col("domain") == "gmail.com")
# display(df_gmail)
df_yahoo = df.filter(col("domain") == "yahoo.com")
df_hotmail = df.filter(col("domain") == "hotmail.com")


## Data Writing

In [0]:
df = df.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))

df.write.format("delta")\
    .mode("overwrite")\
    .save("abfss://silver@storageaccountsunnybest.dfs.core.windows.net/customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customers_silver
USING delta
LOCATION 'abfss://silver@storageaccountsunnybest.dfs.core.windows.net/customers'

In [0]:
%sql
SELECT *
FROM customers_silver;